In [1]:
# Setting up the colaboratory machine
! apt update
! apt upgrade

# This might take few minutes

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu bionic-cran40/ InRelease
Ign:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu1804/x86_64  InRelease
Get:3 http://security.ubuntu.com/ubuntu bionic-security InRelease [88.7 kB]
Ign:4 https://developer.download.nvidia.com/compute/machine-learning/repos/ubuntu1804/x86_64  InRelease
Hit:5 http://archive.ubuntu.com/ubuntu bionic InRelease
Hit:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu1804/x86_64  Release
Hit:7 https://developer.download.nvidia.com/compute/machine-learning/repos/ubuntu1804/x86_64  Release
Hit:8 http://ppa.launchpad.net/c2d4u.team/c2d4u4.0+/ubuntu bionic InRelease
Get:9 http://archive.ubuntu.com/ubuntu bionic-updates InRelease [88.7 kB]
Hit:10 http://ppa.launchpad.net/cran/libgit2/ubuntu bionic InRelease
Hit:11 http://ppa.launchpad.net/deadsnakes/ppa/ubuntu bionic InRelease
Get:12 http://archive.ubuntu.com/ubuntu bionic-backports InRelease [74.6 kB]
Hit:13 http://ppa.launchpad.net/graph

In [2]:
# Installing ASE, GPAW and Chemcoord
# -- ASE: Manage the atoms
# -- GPAW: Does the DFT calculations
# -- Chemcoord: Transformation to internal coordinates
! apt-get install libfftw3-dev libfftw3-doc
! pip install pytest
! pip install ase
! apt-get install python3-mpi4py cython3 libxc-dev gpaw-data
! pip install gpaw
! pip install chemcoord

Reading package lists... Done
Building dependency tree       
Reading state information... Done
libfftw3-dev is already the newest version (3.3.7-1).
libfftw3-doc is already the newest version (3.3.7-1).
The following packages were automatically installed and are no longer required:
  linux-headers-4.15.0-162 linux-headers-4.15.0-162-generic
Use 'apt autoremove' to remove them.
0 upgraded, 0 newly installed, 0 to remove and 4 not upgraded.
Reading package lists... Done
Building dependency tree       
Reading state information... Done
cython3 is already the newest version (0.26.1-0.4).
gpaw-data is already the newest version (0.9.20000-1).
libxc-dev is already the newest version (3.0.0-1build1).
python3-mpi4py is already the newest version (2.0.0-3).
The following packages were automatically installed and are no longer required:
  linux-headers-4.15.0-162 linux-headers-4.15.0-162-generic
Use 'apt autoremove' to remove them.
0 upgraded, 0 newly installed, 0 to remove and 4 not upgraded.


In [3]:
# After making the installation the Runtime must be restarted using Ctrl+M and run the first two cells.
# This process doesn't take long

In [4]:
# Loading the libraries
import chemcoord as cc
import pandas as pd
import numpy as np

In [5]:
# Atoms is used to build a structure
# GPAW is the DFT calculator
from ase import Atoms
from gpaw import GPAW

In [6]:
# Function that receives ASE atoms object and returns the zmat matrix values
# in array form (zmat_values) and the construction table

def get_isomer_zmat(isomer_):
    xyz = cc.Cartesian.from_ase_atoms(isomer_)
    construction_table = xyz.get_construction_table() #Table that contains parameters of transformation
    zmat_matrix = xyz.get_zmat(construction_table)

    Natoms = isomer_.get_number_of_atoms()
    zmat_values = zmat_matrix.iloc[:, 2].tolist()+zmat_matrix.iloc[:, 4].tolist()+zmat_matrix.iloc[:, 6].tolist()
    zmat_values = np.array(zmat_values).reshape(1,3*Natoms)

    return zmat_values, construction_table

In [7]:
# Example using Ag5 isomer defined here "by hand" but to be replaced by a predicition from Flonaco
# -- Building Ag5 isomer, setting the cell and centering the molecule
isomer = Atoms('Ag5', positions=[[8.,        6.738058,  8.000017], [5.316466,  6.774461,  8.], 
                                 [6.629008,  9.13116,   8.000001], [9.370992,  9.13116,   8.000001],
                                 [10.683534,  6.774461,  8.000001]])
cell = [16, 16, 10]
isomer.set_cell(cell)
isomer.center()
# Transform to zmat 
isomer_zarray, c_table = get_isomer_zmat(isomer) # zmat_values (array)

/usr/local/lib/python3.7/dist-packages/ase/atoms.py:968: VisibleDeprecationWarning: Use get_global_number_of_atoms() instead
  np.VisibleDeprecationWarning)


In [8]:
# Function that convert zmat values in array form to zmat matrix for chemcoord
def build_zmat_matrix(zmat_values, construction_table, symbols, Natoms):
    zmat_values = zmat_values.reshape(3, Natoms)
    zmat_matrix = construction_table.copy()
    zmat_matrix.insert(0, "atom", symbols, True) #Adding zmat_values as columns 
    zmat_matrix.insert(2, "bond", zmat_values[0], True)
    zmat_matrix.insert(4, "angle", zmat_values[1], True)
    zmat_matrix.insert(6, "dihedral", zmat_values[2], True)
    zmat_matrix_cc = cc.Zmat(zmat_matrix) # Zmat matrix object
    return zmat_matrix_cc

In [9]:
# Building the zmatrix from array 
isomer_zmat = build_zmat_matrix(isomer_zarray, c_table, isomer.get_chemical_symbols(), isomer.get_number_of_atoms())

/usr/local/lib/python3.7/dist-packages/ase/atoms.py:968: VisibleDeprecationWarning: Use get_global_number_of_atoms() instead
  np.VisibleDeprecationWarning)


In [10]:
# Converting zmat into ASE object using Chemcoord
# Making all the DFT calculations

# Setting the molecule parameters
isomer_ase = isomer_zmat.get_cartesian().get_ase_atoms() #First Cartesian, then builds the molecule
 
isomer_ase.set_cell(cell)
isomer_ase.center()
isomer_ase.set_pbc(True)

# DFT calculator low level precision but faster (takes 1 minute in serial)
calc = GPAW(mode = 'lcao', h =0.2, xc = 'PBE', spinpol = True, nbands = -4)

#DFT calculator with higher precision but takes longer (about 30 minutes in serial).
#calc = GPAW(mode = 'fd', h =0.18, xc = 'PBE', eigensolver = 'rmm-diis', spinpol = True, nbands=-4)

isomer_ase.set_calculator(calc)

# Getting the potential energy
potential_energy = isomer_ase.get_potential_energy()


  ___ ___ ___ _ _ _  
 |   |   |_  | | | | 
 | | | | | . | | | | 
 |__ |  _|___|_____|  21.6.0
 |___|_|             

User:   ???@10de423b4279
Date:   Sat Dec  4 01:27:47 2021
Arch:   x86_64
Pid:    15603
Python: 3.7.12
gpaw:   /usr/local/lib/python3.7/dist-packages/gpaw
_gpaw:  /usr/local/lib/python3.7/dist-packages/
        _gpaw.cpython-37m-x86_64-linux-gnu.so
ase:    /usr/local/lib/python3.7/dist-packages/ase (version 3.22.1)
numpy:  /usr/local/lib/python3.7/dist-packages/numpy (version 1.21.4)
scipy:  /usr/local/lib/python3.7/dist-packages/scipy (version 1.7.3)
libxc:  3.0.0
units:  Angstrom and eV
cores: 1
OpenMP: False
OMP_NUM_THREADS: 1

Input parameters:
  h: 0.2
  mode: lcao
  nbands: -4
  spinpol: True
  xc: PBE

System changes: positions, numbers, cell, pbc, initial_charges, initial_magmoms 

Initialize ...

Ag-setup:
  name: Silver
  id: 33ddeab48f408598355e2011f1241e14
  Z: 47.0
  valence: 17
  core: 30
  charge: 0.0
  file: /usr/share/gpaw-setups/Ag.PBE.gz
  compensatio

In [11]:
print("Potential energy: %.3f"%potential_energy)

Potential energy: -6.137
